In [ ]:
import sqlite3
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# Try to find the correct database file
import os
# db_files = [f for f in os.listdir('.') if f.endswith('.db') or f.endswith('.sqlite') or 'text' in f]
db_path = '/projects/p32143/RL_human_decision/statistic/text_2026-09-22.db'
# if not os.path.exists(db_path) and db_files:
#     db_path = db_files[0] # Fallback to the first db file found

methods = {
    'XXX': 'rl_text_guess',
    'LLM_decision': 'llm_decision_guess',
    'DPO': 'llm_preference_guess',
    'VR-GRPO': 'rlvr_guess'
}

try:
    conn = sqlite3.connect(db_path)
    results = []
    
    for method, table in methods.items():
        # Query the data
        df = pd.read_sql_query(f"SELECT label_truth, guess, correct FROM {table} WHERE guess IS NOT NULL AND guess != ''", conn)
        
        # Standardize strings to lowercase just in case
        df['label_truth'] = df['label_truth'].str.lower()
        df['guess'] = df['guess'].str.lower()
        
        # Accuracies
        overall_acc = df['correct'].mean()
        acc_human = df[df['label_truth'] == 'human']['correct'].mean()
        acc_ai = df[df['label_truth'] == 'ai']['correct'].mean()
        
        # P/R/F1 treating 'ai' as the positive class
        y_true = (df['label_truth'] == 'ai').astype(int)
        y_pred = (df['guess'] == 'ai').astype(int)
        
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        
        results.append({
            'Method': method,
            'Overall Accuracy': f"{overall_acc:.4f}",
            'Acc (Human Text)': f"{acc_human:.4f}",
            'Acc (AI Text)': f"{acc_ai:.4f}",
            'Precision (AI)': f"{precision:.4f}",
            'Recall (AI)': f"{recall:.4f}",
            'F1 Score (AI)': f"{f1:.4f}",
            'Count': len(df)
        })

    results_df = pd.DataFrame(results)
    print(results_df.to_markdown(index=False))
except Exception as e:
    print("Error:", e)
    print("Files in dir:", os.listdir('.'))

| Method       |   Overall Accuracy |   Acc (Human Text) |   Acc (AI Text) |   Precision (AI) |   Recall (AI) |   F1 Score (AI) |   Count |
|:-------------|-------------------:|-------------------:|----------------:|-----------------:|--------------:|----------------:|--------:|
| humanRL      |             0.6469 |             0.6702 |          0.6212 |           0.6314 |        0.6212 |          0.6263 |    1957 |
| LLM_decision |             0.5526 |             0.6166 |          0.4859 |           0.5489 |        0.4859 |          0.5155 |    1084 |
| preferenceRL |             0.5131 |             0.5833 |          0.4379 |           0.4952 |        0.4379 |          0.4648 |    1717 |
| rlvr         |             0.5336 |             0.6089 |          0.4517 |           0.5152 |        0.4517 |          0.4813 |     864 |


In [6]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to the database
# conn = sqlite3.connect('/projects/p32143/RL_human_decision/statistic/text_2026-09-22.db')
conn = sqlite3.connect('/projects/p32143/RL_human_decision/baseline/LLM_decision/gpt56_baseline.db')

# Query the text_guess table
# df = pd.read_sql_query("SELECT * FROM rl_text_guess", conn)
# df = pd.read_sql_query("SELECT * FROM llm_decision_guess", conn)
# df = pd.read_sql_query("SELECT * FROM llm_preference_guess", conn)
# df = pd.read_sql_query("SELECT * FROM rlvr_guess", conn)
df = pd.read_sql_query("SELECT * FROM text_guess", conn)


# Calculate overall accuracy
overall_acc = df['correct'].mean()

# Calculate accuracy by truth label
acc_by_truth = df.groupby('label_truth')['correct'].mean().reset_index()

# Calculate accuracy by explanation source
# acc_by_source = df.groupby('explanation_source')['correct'].agg(['mean', 'count']).reset_index().rename(columns={'mean': 'accuracy', 'count': 'trials'})

# Calculate guess distribution by explanation source and truth label
# guess_dist = df.groupby(['explanation_source', 'label_truth'])['guess'].value_counts(normalize=True).unstack().reset_index()

# import matplotlib.pyplot as plt

# plt.figure(figsize=(12, 6))
# sns.barplot(data=df, x='explanation_source', y='correct', hue='label_truth')
# plt.xticks(rotation=45, ha='right')
# plt.title('Human Guess Accuracy by Explanation Source and Ground Truth')
# plt.tight_layout()
# plt.savefig('accuracy_by_source.png')

print(f"Overall Accuracy: {overall_acc:.2%}")
print("\nAccuracy by Truth Label:")
print(acc_by_truth)
# print("\nAccuracy by Explanation Source:")
# print(acc_by_source.sort_values('accuracy', ascending=False))

# print(pd.crosstab(df['explanation_source'], df['label_truth']))

# conf_matrix = pd.crosstab(df['label_truth'], df['guess'], normalize='index') * 100
# print("Confusion Matrix (Row %):")
# print(conf_matrix)

# # Overall AI prediction rate for AI texts vs Human texts
# print("\nGuessing 'AI' percentage by truth:")
# print(df.groupby('label_truth').apply(lambda x: (x['guess'] == 'AI').mean() * 100))

# # Let's see the % of 'AI' guesses for each explanation source
# ai_guess_rate = df.groupby(['label_truth', 'explanation_source']).apply(lambda x: (x['guess'] == 'AI').mean() * 100).reset_index(name='% Guessed AI')
# print(ai_guess_rate)

# print(f"Number of unique text_ids: {df['text_id'].nunique()}")
# print(f"Number of unique users: {df['user_id'].nunique()}")
# print(f"Total trials: {len(df)}")

# plt.figure(figsize=(10, 6))

# # Clean up explanation source names for the chart
# df_plot = df.copy()
# df_plot['explanation_type'] = df_plot['explanation_source'].apply(lambda x: 'Gemini' if 'gemini' in x else ('Claude' if 'claude' in x else 'Original/Human'))

# # Plot the % of "AI" guesses by truth label and explanation type
# ai_guess_rate = df_plot.groupby(['label_truth', 'explanation_type']).apply(lambda x: (x['guess'] == 'AI').mean() * 100).reset_index(name='percent_ai')

# sns.barplot(data=ai_guess_rate, x='label_truth', y='percent_ai', hue='explanation_type', 
#             palette={'Original/Human': '#2ca02c', 'Claude': '#ff7f0e', 'Gemini': '#1f77b4'},
#             order=['Human', 'AI'])

# plt.title('Percentage of "AI" Guesses by Ground Truth and Explanation Type', fontsize=14)
# plt.ylabel('% Guessed as AI', fontsize=12)
# plt.xlabel('True Label of Text', fontsize=12)
# plt.ylim(0, 60)
# plt.legend(title='Explanation Source')
# plt.grid(axis='y', linestyle='--', alpha=0.7)
# plt.tight_layout()
# plt.savefig('ai_guess_rate.png')

conn.close()

Overall Accuracy: 48.80%

Accuracy by Truth Label:
  label_truth   correct
0          AI  0.000000
1       Human  0.965298


In [3]:
import sqlite3
import pandas as pd

# Re-connect to the database
conn = sqlite3.connect('/projects/p32143/RL_human_decision/statistic/text_2026-09-02.db')
df = pd.read_sql_query("SELECT text_id, label_truth, explanation_source FROM text_guess", conn)

# Group by text_id and label_truth, then count unique explanation sources
explanation_counts = df.groupby(['text_id', 'label_truth'])['explanation_source'].nunique().reset_index()

# Filter for texts that have exactly two explanations
two_explanations = explanation_counts[explanation_counts['explanation_source'] == 2]

# Count how many of these fall under AI vs Human
result = two_explanations['label_truth'].value_counts()

print("Texts with exactly two explanation sources:")
print(result)

# Just to be thorough, let's also see the distribution of explanation counts overall
print("\nDistribution of explanation counts per text/label combination:")
print(explanation_counts['explanation_source'].value_counts())

conn.close()

Texts with exactly two explanation sources:
label_truth
Human    768
AI       752
Name: count, dtype: int64

Distribution of explanation counts per text/label combination:
explanation_source
1    2195
2    1520
Name: count, dtype: int64


In [2]:
import sqlite3
import pandas as pd

# Re-connect to the database
conn = sqlite3.connect('/projects/p32143/RL_human_decision/statistic/text_2026-09-02.db')
df = pd.read_sql_query("SELECT text_id, label_truth FROM text_guess", conn)

# Calculate unique text_ids for each label
unique_ai_texts = df[df['label_truth'] == 'AI']['text_id'].nunique()
unique_human_texts = df[df['label_truth'] == 'Human']['text_id'].nunique()
total_unique_versions = df[['text_id', 'label_truth']].drop_duplicates().shape[0]

print(f"Unique texts with AI label: {unique_ai_texts}")
print(f"Unique texts with Human label: {unique_human_texts}")
print(f"Total unique text versions: {total_unique_versions}")

conn.close()

Unique texts with AI label: 1838
Unique texts with Human label: 1877
Total unique text versions: 3715


In [1]:
import sqlite3
import pandas as pd

# 1. Connect to your SQLite database
conn = sqlite3.connect('/projects/p32143/RL_human_decision/statistic/text_2026-09-02.db') # Update with your actual file path

# 2. Load the relevant columns into a DataFrame
df = pd.read_sql_query("SELECT label_truth, guess FROM text_guess", conn)

# 3. Clean the data (standardize to lowercase to catch any formatting inconsistencies)
df['label_truth'] = df['label_truth'].astype(str).str.strip().str.lower()
df['guess'] = df['guess'].astype(str).str.strip().str.lower()

# 4. Filter into Human and AI subsets
human_subset = df[df['label_truth'] == 'human']
ai_subset = df[df['label_truth'] == 'ai']

# 5. Calculate accuracies
human_accuracy = (human_subset['label_truth'] == human_subset['guess']).mean()
ai_accuracy = (ai_subset['label_truth'] == ai_subset['guess']).mean()
overall_accuracy = (df['label_truth'] == df['guess']).mean()

print(f"Overall Accuracy: {overall_accuracy:.2%}")
print(f"Human Text Accuracy: {human_accuracy:.2%} (N={len(human_subset)})")
print(f"AI Text Accuracy: {ai_accuracy:.2%} (N={len(ai_subset)})")

conn.close()

Overall Accuracy: 51.28%
Human Text Accuracy: 61.17% (N=10374)
AI Text Accuracy: 41.16% (N=10146)


In [4]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect('/projects/p32143/RL_human_decision/statistic/text_2026-09-02.db')
df = pd.read_sql_query("SELECT user_id, text_id, label_truth, explanation_source FROM text_guess", conn)

# 1. Average number of text+explanation combinations assigned to each human
# Group by user_id and count unique combinations of (text_id, label_truth, explanation_source)
user_counts = df.groupby('user_id').apply(lambda x: x[['text_id', 'label_truth', 'explanation_source']].drop_duplicates().shape[0])
avg_texts_per_human = user_counts.mean()

# 2. Average number of human decisions each text+explanation combination gets
# Group by (text_id, label_truth, explanation_source) and count the number of decisions (rows)
decision_counts = df.groupby(['text_id', 'label_truth', 'explanation_source']).size()
avg_decisions_per_text = decision_counts.mean()

print(f"Average text+explanation combinations per human: {avg_texts_per_human:.2f}")
print(f"Average human decisions per text+explanation combination: {avg_decisions_per_text:.2f}")

# Extra detail just to be helpful: min/max
print(f"Min/Max per human: {user_counts.min()} / {user_counts.max()}")
print(f"Min/Max per text+explanation: {decision_counts.min()} / {decision_counts.max()}")

conn.close()

Average text+explanation combinations per human: 8.67
Average human decisions per text+explanation combination: 3.92
Min/Max per human: 1 / 2600
Min/Max per text+explanation: 3 / 7


In [18]:
import sys
!module load R/4.4.0 && {sys.executable} -m pip install rpy2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymer4.models import Lmer # True Mixed-Effects modeling

# 1. Project Parameters
base_texts = 3715
acc_base_h = 0.6117
acc_base_a = 0.4116
acc_target = 0.7103 # Simulator target

# X-Axis: Number of texts evaluated in deployment (from 20 up to 200)
deployment_sample_sizes = range(20, 220, 20)
m_trials = 100 # Keep low for testing, GLMMs are computationally heavy

power_results = []

print("Running True GLMM Mixed-Effects Power Analysis...")

for n_deploy in deployment_sample_sizes:
    significant_count = 0
    
    for _ in range(m_trials):
        # A. Simulate a subset of the massive baseline
        # (We use a subset here just to speed up the GLMM computation)
        base_h = np.random.binomial(1, acc_base_h, 500)
        base_a = np.random.binomial(1, acc_base_a, 500)
        
        df_base = pd.DataFrame({
            'outcome': np.concatenate([base_h, base_a]),
            'treatment': 0,
            'text_id': np.random.randint(0, base_texts, 1000),
            'human_id': np.random.randint(0, 200, 1000) # Simulating ~5 decisions per human
        })
        
        # B. Simulate the Deployment (1 decision per text for n_deploy texts)
        deploy_h = np.random.binomial(1, acc_target, n_deploy // 2)
        deploy_a = np.random.binomial(1, acc_target, n_deploy // 2)
        
        df_deploy = pd.DataFrame({
            'outcome': np.concatenate([deploy_h, deploy_a]),
            'treatment': 1,
            'text_id': np.random.randint(base_texts, base_texts + n_deploy, n_deploy),
            'human_id': np.random.randint(200, 250, n_deploy) # New evaluators
        })
        
        df_total = pd.concat([df_base, df_deploy])
        
        # C. Fit True Binomial GLMM with random intercepts for human and text
        # Formula: outcome predicted by treatment, with random effects for human_id and text_id
        model = Lmer('outcome ~ treatment + (1|human_id) + (1|text_id)', data=df_total, family='binomial')
        
        try:
            # Fit the model silently
            model.fit(summarize=False, verbose=False)
            
            # Check if the treatment effect is significant
            if model.coefs.loc['treatment', 'P-val'] < 0.05:
                significant_count += 1
        except Exception:
            continue
            
    power = significant_count / m_trials
    power_results.append(power)
    print(f"Deployment Size: {n_deploy} | Power: {power:.2%}")

# 2. Plotting the realistic curve
plt.figure(figsize=(10, 6))
plt.plot(deployment_sample_sizes, power_results, marker='o', color='teal', linewidth=2)
plt.axhline(y=0.80, color='red', linestyle='--', label='80% Power Threshold')
plt.title('True GLMM Power Curve (Focusing on <200 decisions)', fontsize=14)
plt.xlabel('Number of Deployment Texts Evaluated (1 decision per text)', fontsize=12)
plt.ylabel('Statistical Power', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

  Using cached rpy2-3.6.7-py3-none-any.whl.metadata (5.5 kB)
  Using cached rpy2_rinterface-3.6.6.tar.gz (81 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached rpy2_robjects-3.6.5-py3-none-any.whl.metadata (3.2 kB)
  Using cached cffi-2.1.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.5 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached rpy2-3.6.7-py3-none-any.whl (9.9 kB)
Using cached cffi-2.1.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (221 kB)
Using cached rpy2_robjects-3.6.5-py3-none-any.whl (125 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)
  Created wheel for rpy2-rinterface: filename=rpy2_rinterface-3.6.6-cp312-cp312-linux_x86_64.whl size=140510 sha256=ef0bf216837a05382e2b773a88b08de79799fba6f17d16a9262eb901d7b6cdd8
  Stored in directory: /gpfs/home/kjj8053/.cache/pip/wheels/33/80/32/63ab71b1fb32

Unable to determine R home: [Errno 2] No such file or directory: 'R'
Error importing in API mode: ImportError('libiconv.so.2: cannot open shared object file: No such file or directory')
Trying to import in ABI mode.


ValueError: r_home is None. Try python -m rpy2.situation

In [14]:
!module load R/4.4.0 && pip install rpy2

Defaulting to user installation because normal site-packages is not writeable
  Using cached rpy2-3.6.7-py3-none-any.whl.metadata (5.5 kB)
  Using cached rpy2_rinterface-3.6.6.tar.gz (81 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached cffi-2.0.0-cp39-cp39-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.6 kB)
  Using cached pycparser-2.23-py3-none-any.whl.metadata (993 bytes)
Using cached cffi-2.0.0-cp39-cp39-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (216 kB)
Using cached pycparser-2.23-py3-none-any.whl (118 kB)
  Created wheel for rpy2-rinterface: filename=rpy2_rinterface-3.6.6-cp39-cp39-linux_x86_64.whl size=236068 sha256=127e211f1d774b1acd769b019f3141c02bb87a6083cdf39cedcb118210639fa9
  Stored in directory: /gpfs/home/kjj8053/.cache/pip/wheels/75/8f/b9/7dd090c2310988ec23257fea1b51005327927db72ebe17e3b3
Successfully built rpy2-rinterface

[notice] A new release of 